<a href="https://colab.research.google.com/github/sidneiclementenobre/-p0-triador-feedbacks-gemini/blob/main/P0_Triador_Feedback_Gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Célula 1: Instalação
!pip install -q google-genai pydantic


In [4]:
# Célula 2: Estrutura de Dados e Serviço com Tratamento de Erros
from typing import List
from google import genai
from google.genai import types
from google.genai.errors import APIError
from google.colab import userdata
from pydantic import BaseModel, Field

class FeedbackAnalise(BaseModel):
    sentimento: str = Field(
        description="Deve ser estritamente: Positivo, Neutro ou Negativo"
    )
    pontos_criticos: List[str] = Field(
        description="Lista objetiva dos problemas, elogios ou pontos importantes citados"
    )
    resumo_executivo: str = Field(
        description="Um resumo de uma única frase sobre o feedback do cliente"
    )

def analisar_feedback(texto_cliente: str) -> FeedbackAnalise:
    try:
        api_key = userdata.get("GEMINI_API_KEY")
    except userdata.SecretNotFoundError:
        raise ValueError("Erro: Segredo 'GEMINI_API_KEY' não encontrado nos Secrets do Colab.")
    except userdata.NotebookAccessError:
        raise PermissionError("Erro: Permissão de acesso ao segredo 'GEMINI_API_KEY' desativada.")

    if not texto_cliente or not texto_cliente.strip():
        raise ValueError("Erro: O texto do feedback não pode estar vazio.")

    client = genai.Client(api_key=api_key, http_options={"api_version": "v1beta"})

    prompt_sistema = (
        "Você é um especialista em atendimento ao cliente e análise de dados. "
        "Leia o feedback do cliente, classifique o sentimento predominante e "
        "extraia os pontos críticos de forma clara e objetiva."
    )

    try:
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            contents=texto_cliente,
            config=types.GenerateContentConfig(
                system_instruction=prompt_sistema,
                response_mime_type="application/json",
                response_schema=FeedbackAnalise,
                temperature=0.1,
            ),
        )
        return FeedbackAnalise.model_validate_json(response.text)

    except APIError as e:
        print(f"Erro na comunicação com a API do Gemini: {e}")
        raise
    except Exception as e:
        print(f"Erro inesperado ao processar o feedback: {e}")
        raise
